# Baseline Anomaly Classification

This notebook builds the first baseline ML model for SIH26073.

Goal:
Classify AWS weather observations into:
- NORMAL
- SENSOR_FAULT
- TRANSMISSION_GLITCH
- GENUINE_EVENT

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
current_path = Path.cwd()

project_root = None

for path in [current_path] + list(current_path.parents):
    if (path / "01_ml").exists():
        project_root = path
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not find the Skyguard_AI project folder."
    )

print("Project root:", project_root)

Project root: c:\Users\sanch\Documents\Git_hub\Skyguard_AI


### Project root
- Here we will use the same approach we used in loader / EDA.

In [3]:
data_path = (
    project_root
    / "01_ml"
    / "01_data"
    / "03_synthetic"
    / "aws_weather_anomaly_dataset_v2_15col.csv"
)

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully.
Rows: 20883
Columns: 25


- Load anomaly dataset

In [4]:
data_path = (
    project_root
    / "01_ml"
    / "01_data"
    / "03_synthetic"
    / "aws_weather_anomaly_dataset_v2_15col.csv"
)

df = pd.read_csv(data_path)

print("Dataset loaded successfully.")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully.
Rows: 20883
Columns: 25


- Inspect columns : 

Yahan humein verify karna hai ki weather columns ke saath ground-truth columns bhi present hain.

In [5]:
print("Columns:\n")

for column in df.columns:
    print("-", column)

Columns:

- cluster
- station_name
- state
- district
- date_of_record
- avg_temp
- min_temp
- max_temp
- relative_humidity
- wind_speed
- air_pressure
- rainfall
- elevation
- latitude
- longitude
- is_injected
- anomaly_category
- anomaly_type
- affected_feature
- anomaly_id
- event_start
- event_end
- severity
- original_value
- modified_value


- Target distribution

In [6]:
print("\nAnomaly category distribution:")
print(df["anomaly_category"].value_counts())

print("\nPercentage distribution:")
print(
    (df["anomaly_category"].value_counts(normalize=True) * 100)
    .round(2)
)


Anomaly category distribution:
anomaly_category
NORMAL                 19955
SENSOR_FAULT             718
GENUINE_EVENT            120
TRANSMISSION_GLITCH       90
Name: count, dtype: int64

Percentage distribution:
anomaly_category
NORMAL                 95.56
SENSOR_FAULT            3.44
GENUINE_EVENT           0.57
TRANSMISSION_GLITCH     0.43
Name: proportion, dtype: float64


- Missing values

In [7]:
missing_values = df.isna().sum()

print("Missing values:")
print(missing_values[missing_values > 0])

Missing values:
avg_temp                 3
min_temp                 9
max_temp                 3
relative_humidity        7
wind_speed              38
air_pressure            47
rainfall                25
anomaly_type         19955
affected_feature     19955
anomaly_id           19955
event_start          19955
event_end            19955
severity             19955
original_value       19975
modified_value       20005
dtype: int64


- Ground-truth sanity check

In [8]:
print("Injected rows:", df["is_injected"].sum())
print("Normal rows:", (df["is_injected"] == False).sum())

print("\nInjected rows by category:")
print(
    df.loc[df["is_injected"] == True, "anomaly_category"]
    .value_counts()
)

Injected rows: 928
Normal rows: 19955

Injected rows by category:
anomaly_category
SENSOR_FAULT           718
GENUINE_EVENT          120
TRANSMISSION_GLITCH     90
Name: count, dtype: int64


In [9]:
df.head()

,cluster,station_name,state,district,date_of_record,avg_temp,min_temp,max_temp,relative_humidity,wind_speed,...,is_injected,anomaly_category,anomaly_type,affected_feature,anomaly_id,event_start,event_end,severity,original_value,modified_value
0,Punjab Plains,Ambala,HR,Ambala,2021-01-01,9.1,4.4,13.6,78.400208,NaN,...,False,NORMAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Punjab Plains,Ambala,HR,Ambala,2021-01-02,11.8,5.7,14.6,75.500032,5.8,...,False,NORMAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Punjab Plains,Ambala,HR,Ambala,2021-01-03,14.6,9.4,20.0,73.037773,11.5,...,False,NORMAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Punjab Plains,Ambala,HR,Ambala,2021-01-04,18.1,11.2,23.2,64.934059,9.1,...,False,NORMAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Punjab Plains,Ambala,HR,Ambala,2021-01-05,16.2,14.5,24.1,67.400295,13.2,...,False,NORMAL,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


---

### Step 2 — Define model features and target

In [10]:
# Features that will be given to the ML model
feature_cols = [
    "avg_temp",
    "min_temp",
    "max_temp",
    "relative_humidity",
    "wind_speed",
    "air_pressure",
    "rainfall",
    "elevation",
    "latitude",
    "longitude",
]

# Target column
target_col = "anomaly_category"

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Number of features:", len(feature_cols))
print("\nFeatures:")
print(feature_cols)

print("\nTarget column:", target_col)

print("\nX shape:", X.shape)
print("y shape:", y.shape)

Number of features: 10

Features:
['avg_temp', 'min_temp', 'max_temp', 'relative_humidity', 'wind_speed', 'air_pressure', 'rainfall', 'elevation', 'latitude', 'longitude']

Target column: anomaly_category

X shape: (20883, 10)
y shape: (20883,)


df
│
├── Weather features ──→ X
│
└── anomaly_category ──→ y

- Missing values quick check before train/ test split

In [11]:
print("Missing values in model features:")

missing_features = X.isna().sum()

print(
    missing_features[missing_features > 0]
)

Missing values in model features:
avg_temp              3
min_temp              9
max_temp              3
relative_humidity     7
wind_speed           38
air_pressure         47
rainfall             25
dtype: int64


- Train/Test split

In [ ]:
# from sklearn.model_selection import train_test_split

# X_train, X_test, y_train, y_test = train_test_split(
#     X,
#     y,
#     test_size=0.20,
#     random_state=42,
#     stratify=y
# )

# print("Training data:", X_train.shape)
# print("Testing data :", X_test.shape)

# print("\nTraining class distribution:")
# print(y_train.value_counts())

# print("\nTesting class distribution:")
# print(y_test.value_counts())

- Note : Hamare pass har station ka 2021 se 2025 tak ka data hai.

#### Date range check

In [13]:
print("Date range:")

print(
    "Start:",
    pd.to_datetime(df["date_of_record"]).min().date()
)

print(
    "End  :",
    pd.to_datetime(df["date_of_record"]).max().date()
)

print("\nNumber of unique dates:")

print(
    pd.to_datetime(df["date_of_record"]).nunique()
)

Date range:
Start: 2021-01-01
End  : 2025-02-10

Number of unique dates:
1502


- df ki date column ko proper datetime bana denge

In [14]:
df["date_of_record"] = pd.to_datetime(
    df["date_of_record"],
    errors="coerce"
)

print("Date column type:", df["date_of_record"].dtype)
print("Missing dates:", df["date_of_record"].isna().sum())

Date column type: datetime64[us]
Missing dates: 0


- date-range wala cell original simple form mein bhi chalega:

In [15]:
print("Overall date range:")
print("Start:", df["date_of_record"].min().date())
print("End  :", df["date_of_record"].max().date())

print("\nNumber of unique dates:")
print(df["date_of_record"].nunique())

Overall date range:
Start: 2021-01-01
End  : 2025-02-10

Number of unique dates:
1502


- split dates

In [16]:
# Get all unique dates in chronological order
unique_dates = sorted(
    df["date_of_record"].unique()
)

total_dates = len(unique_dates)

# 70% for training
train_end = int(total_dates * 0.70)

# Next 15% for validation
validation_end = int(total_dates * 0.85)

train_end_date = unique_dates[train_end - 1]
validation_end_date = unique_dates[validation_end - 1]

print("Train ends on      :", train_end_date)
print("Validation ends on :", validation_end_date)
print("Test starts after  :", validation_end_date)

Train ends on      : 2023-11-17 00:00:00
Validation ends on : 2024-06-29 00:00:00
Test starts after  : 2024-06-29 00:00:00


- Actual 3 datasets created

In [17]:
# Create chronological train, validation, and test sets

train_df = df[
    df["date_of_record"] <= train_end_date
].copy()

validation_df = df[
    (df["date_of_record"] > train_end_date)
    & (df["date_of_record"] <= validation_end_date)
].copy()

test_df = df[
    df["date_of_record"] > validation_end_date
].copy()


print("Train rows      :", len(train_df))
print("Validation rows :", len(validation_df))
print("Test rows       :", len(test_df))
print("Total rows      :", len(train_df) + len(validation_df) + len(test_df))

Train rows      : 14689
Validation rows : 3148
Test rows       : 3046
Total rows      : 20883


- 3-way chronological split successfully ho gaya

In [18]:
print("TRAIN class distribution:")
print(train_df["anomaly_category"].value_counts())
print()

print("VALIDATION class distribution:")
print(validation_df["anomaly_category"].value_counts())
print()

print("TEST class distribution:")
print(test_df["anomaly_category"].value_counts())

TRAIN class distribution:
anomaly_category
NORMAL                 13999
SENSOR_FAULT             536
GENUINE_EVENT             90
TRANSMISSION_GLITCH       64
Name: count, dtype: int64

VALIDATION class distribution:
anomaly_category
NORMAL                 3019
SENSOR_FAULT            107
TRANSMISSION_GLITCH      13
GENUINE_EVENT             9
Name: count, dtype: int64

TEST class distribution:
anomaly_category
NORMAL                 2937
SENSOR_FAULT             75
GENUINE_EVENT            21
TRANSMISSION_GLITCH      13
Name: count, dtype: int64


- training set ke missing values

In [19]:
# Training features only
X_train = train_df[feature_cols].copy()

print("Training feature shape:", X_train.shape)

Training feature shape: (14689, 10)


In [20]:
print("Missing values in training features:")

train_missing = X_train.isna().sum()

print(
    train_missing[train_missing > 0]
)

Missing values in training features:
avg_temp              3
min_temp              5
max_temp              2
relative_humidity     3
wind_speed           10
air_pressure         13
rainfall             15
dtype: int64


- validation/test

In [21]:
# Create features for validation and test sets
X_val = validation_df[feature_cols].copy()
X_test = test_df[feature_cols].copy()

# Create targets for all three sets
y_train = train_df[target_col].copy()
y_val = validation_df[target_col].copy()
y_test = test_df[target_col].copy()

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

X_train: (14689, 10)
X_val  : (3148, 10)
X_test : (3046, 10)

y_train: (14689,)
y_val  : (3148,)
y_test : (3046,)


##### Step  — Fitting IMPUTER only upon Training Data

- Hum abhi Median Imputer use karenge. Weather measurements mein kuch naturally high/low values ho sakti hain, isliye median mean se zyada robust starting choice hai.

In [22]:
from sklearn.impute import SimpleImputer

# Create the imputer
imputer = SimpleImputer(strategy="median")

# Learn the median values from training data only
imputer.fit(X_train)    # This thing is very improtant to note 

print("Imputer fitted successfully.")
print("\nMedian values learned from training data:")

for feature, value in zip(feature_cols, imputer.statistics_):
    print(f"{feature}: {value:.4f}")

Imputer fitted successfully.

Median values learned from training data:
avg_temp: 25.8000
min_temp: 22.2000
max_temp: 30.2000
relative_humidity: 72.4000
wind_speed: 7.4000
air_pressure: 1009.7000
rainfall: 0.1000
elevation: 271.0000
latitude: 30.3833
longitude: 76.7667


- Step — Imputation

In [23]:
# Apply the training-fitted imputer to all three datasets

X_train_imputed = imputer.transform(X_train)
X_val_imputed = imputer.transform(X_val)
X_test_imputed = imputer.transform(X_test)

# Convert the results back to DataFrames
X_train_imputed = pd.DataFrame(
    X_train_imputed,
    columns=feature_cols,
    index=X_train.index
)

X_val_imputed = pd.DataFrame(
    X_val_imputed,
    columns=feature_cols,
    index=X_val.index
)

X_test_imputed = pd.DataFrame(
    X_test_imputed,
    columns=feature_cols,
    index=X_test.index
)

print("Imputation completed successfully.")

print("\nMissing values after imputation:")
print("Train:", X_train_imputed.isna().sum().sum())
print("Validation:", X_val_imputed.isna().sum().sum())
print("Test:", X_test_imputed.isna().sum().sum())

Imputation completed successfully.

Missing values after imputation:
Train: 0
Validation: 0
Test: 0


#### Step - Fitting the scaler to standardize the values

- avg_temp       → around 25
- air_pressure   → around 1000
- rainfall       → around 0–100
- latitude       → around 10–35

- For neural network standardized input within a range of -1 to 1 is better for generalizing the pattern .

In [24]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Learn scaling parameters from training data only
scaler.fit(X_train_imputed)

print("Scaler fitted successfully.")

Scaler fitted successfully.


- Applying the scaling

In [25]:
# Apply the training-fitted scaler to all three datasets

X_train_scaled = scaler.transform(X_train_imputed)
X_val_scaled = scaler.transform(X_val_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

print("Scaling completed successfully.")

print("\nScaled data shapes:")
print("Train:", X_train_scaled.shape)
print("Validation:", X_val_scaled.shape)
print("Test:", X_test_scaled.shape)

Scaling completed successfully.

Scaled data shapes:
Train: (14689, 10)
Validation: (3148, 10)
Test: (3046, 10)


- Scaling Check

In [26]:
# Check whether the training features are properly standardized

train_means = X_train_scaled.mean(axis=0)
train_stds = X_train_scaled.std(axis=0)

print("Training feature means:")
print(np.round(train_means, 4))

print("\nTraining feature standard deviations:")
print(np.round(train_stds, 4))

Training feature means:
[ 0. -0.  0. -0. -0. -0. -0.  0.  0.  0.]

Training feature standard deviations:
[1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]


---

- Now X_train_scaled, X_val_scaled, X_test_scaled is ready to feed into Tensorflow Model

Next step — target labels assignment is test split

Currently there is text in y mein:

NORMAL
SENSOR_FAULT
TRANSMISSION_GLITCH
GENUINE_EVENT


We will assign numeric labels of these classes to Tensorflow:

NORMAL               → 0
SENSOR_FAULT         → 1
TRANSMISSION_GLITCH  → 2
GENUINE_EVENT        → 3

Lekin mapping khud se hard-code nahi karenge;
We will use label encoder so that same mapping happens on train/validation/test remains consistent throughout "y"

- Step - Encode target labels

In [27]:
from sklearn.preprocessing import LabelEncoder

# Create the label encoder
label_encoder = LabelEncoder()

# Learn the class mapping from training labels only
y_train_encoded = label_encoder.fit_transform(y_train)

# Apply the same mapping to validation and test labels
y_val_encoded = label_encoder.transform(y_val)
y_test_encoded = label_encoder.transform(y_test)

print("Class mapping:")

for number, class_name in enumerate(label_encoder.classes_):
    print(f"{number} -> {class_name}")

Class mapping:
0 -> GENUINE_EVENT
1 -> NORMAL
2 -> SENSOR_FAULT
3 -> TRANSMISSION_GLITCH


Preprocessing Done:

        Raw features
            ↓
        Train/Val/Test chronological split 
            ↓
        Median imputation 
            ↓
        Standard scaling 
            ↓
        Target encoding 
            ↓
        TensorFlow/Keras model    (Next Task)

#### Data split + preprocessing + label encoding complete

- Now we are moving to a separate notebook for performing the training 

--- 

In [28]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.21.0


- Creating the Scope of Reproducibility

In [29]:
# Make model training more reproducible
np.random.seed(42)
tf.random.set_seed(42)

print("Random seeds set successfully.")

Random seeds set successfully.


- Ab next cell = class weights. Ye hamare imbalanced classes ke liye important hai.

In [30]:
from sklearn.utils.class_weight import compute_class_weight

# Find the unique classes in the training data
classes = np.unique(y_train_encoded)

# Calculate balanced class weights
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_encoded
)

# Convert the result into a dictionary
class_weights = {
    int(class_label): float(weight)
    for class_label, weight in zip(classes, class_weights_array)
}

print("Class weights:")

for class_label, weight in class_weights.items():
    class_name = label_encoder.inverse_transform([class_label])[0]

    print(
        f"{class_label} -> {class_name}: {weight:.4f}"
    )

Class weights:
0 -> GENUINE_EVENT: 40.8028
1 -> NORMAL: 0.2623
2 -> SENSOR_FAULT: 6.8512
3 -> TRANSMISSION_GLITCH: 57.3789


Ek important caution

57.38 aur 40.80 kaafi high weights hain. Ye mathematical error nahi hai, but model training mein rare classes ke mistakes ko bahut strongly punish karega. Iska result kabhi-kabhi ye ho sakta hai ki model rare anomalies ko over-predict karne lage.

Isliye hum pehle baseline model ko class weights ke saath train karenge, phir validation precision/recall/F1 dekhenge. Zarurat padi toh weighting strategy refine karenge.

In [31]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

print("Keras imports successful.")

Keras imports successful.


- Building the TensorFlow/Keras model

In [32]:
# Building the baseline neural network

model = Sequential([
    Dense(64, activation="relu", input_shape=(10,)),
    Dense(32, activation="relu"),
    Dropout(0.20),
    Dense(4, activation="softmax")
])

model.summary()

c:\Users\sanch\Documents\Git_hub\Skyguard_AI\AWS_venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,916 (11.39 KB)

 Trainable params: 2,916 (11.39 KB)

 Non-trainable params: 0 (0.00 B)

In [33]:
# Compiling the model

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

Model compiled successfully.


Why sparse_categorical_crossentropy ?

OUR targets:

0
1
2
3

We have numeric class labels, not one-hot encoded. That's why this loss function is best suitable.

Ab actual training

- Hum model.fit() mein:

- training data denge
- validation data denge
- class weights use karenge
- around 30 epochs rakhenge
- EarlyStopping use karenge so model unnecessarily train na karta rahe

In [34]:
from tensorflow.keras.callbacks import EarlyStopping

# Stop training when validation performance stops improving
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

print("Early stopping callback created.")

Early stopping callback created.


- Model training

In [35]:
# Train the model

history = model.fit(
    X_train_scaled,
    y_train_encoded,
    validation_data=(
        X_val_scaled,
        y_val_encoded
    ),
    epochs=30,
    batch_size=32,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4669 - loss: 1.2240 - val_accuracy: 0.1560 - val_loss: 1.3747
Epoch 2/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3415 - loss: 1.0360 - val_accuracy: 0.1722 - val_loss: 1.3154
Epoch 3/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3145 - loss: 0.9922 - val_accuracy: 0.1633 - val_loss: 1.2934
Epoch 4/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3321 - loss: 0.9296 - val_accuracy: 0.1747 - val_loss: 1.3083
Epoch 5/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.3443 - loss: 0.9026 - val_accuracy: 0.1928 - val_loss: 1.2828
Epoch 6/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.3283 - loss: 0.8720 - val_accuracy: 0.2773 - val_loss: 1.1855
Epoch 7/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.3571 - loss: 0.8507 - val_accuracy: 0.2913 - val_loss: 1.1753
Epoch 8/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.3609 - loss: 0.8549 - val_accuracy: 0.

- VALIDATION Prediction Check

In [36]:
from sklearn.metrics import confusion_matrix

# Get model predictions on validation data
val_probabilities = model.predict(
    X_val_scaled,
    verbose=0
)

val_predictions = np.argmax(
    val_probabilities,
    axis=1
)

print("Validation prediction distribution:")
print(
    pd.Series(val_predictions).value_counts().sort_index()
)

print("\nActual validation distribution:")
print(
    pd.Series(y_val_encoded).value_counts().sort_index()
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_val_encoded,
        val_predictions
    )
)

Validation prediction distribution:
0     136
1    1376
2     943
3     693
Name: count, dtype: int64

Actual validation distribution:
0       9
1    3019
2     107
3      13
Name: count, dtype: int64

Confusion matrix:
[[   7    0    1    1]
 [ 124 1328  907  660]
 [   4   40   33   30]
 [   1    8    2    2]]


Matlab model ko rare-class mistakes ki penalty bahut high di gayi. In particular TRANSMISSION_GLITCH ki weight ~57 hai, jabki NORMAL ki ~0.26.

Result:

Model ne rare classes ko miss na karne ke chakkar mein normal observations ko hi anomalies samajhna start kar diya.

---

In [37]:
# Build a second baseline model without class weights

model_no_weights = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(10,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(4, activation="softmax")
])

model_no_weights.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Second baseline model created.")

Second baseline model created.


In [38]:
# Train the second baseline model without class weights

history_no_weights = model_no_weights.fit(
    X_train_scaled,
    y_train_encoded,
    validation_data=(
        X_val_scaled,
        y_val_encoded
    ),
    epochs=30,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9454 - loss: 0.2859 - val_accuracy: 0.9590 - val_loss: 0.2030
Epoch 2/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9532 - loss: 0.2204 - val_accuracy: 0.9590 - val_loss: 0.1983
Epoch 3/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9530 - loss: 0.2101 - val_accuracy: 0.9590 - val_loss: 0.1975
Epoch 4/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9540 - loss: 0.2009 - val_accuracy: 0.9590 - val_loss: 0.1955
Epoch 5/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9553 - loss: 0.1958 - val_accuracy: 0.9590 - val_loss: 0.1969
Epoch 6/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9560 - loss: 0.1911 - val_accuracy: 0.9590 - val_loss: 0.1933
Epoch 7/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9566 - loss: 0.1888 - val_accuracy: 0.9584 - val_loss: 0.1951
Epoch 8/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9569 - loss: 0.1877 - val_accuracy: 0.

In [39]:
from sklearn.metrics import confusion_matrix

# Predict classes for the validation set
val_probabilities_no_weights = model_no_weights.predict(
    X_val_scaled,
    verbose=0
)

val_predictions_no_weights = np.argmax(
    val_probabilities_no_weights,
    axis=1
)

print("Validation prediction distribution:")
print(
    pd.Series(
        val_predictions_no_weights
    ).value_counts().sort_index()
)

print("\nActual validation distribution:")
print(
    pd.Series(
        y_val_encoded
    ).value_counts().sort_index()
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_val_encoded,
        val_predictions_no_weights
    )
)

Validation prediction distribution:
0      14
1    3130
2       4
Name: count, dtype: int64

Actual validation distribution:
0       9
1    3019
2     107
3      13
Name: count, dtype: int64

Confusion matrix:
[[   4    5    0    0]
 [   8 3011    0    0]
 [   2  101    4    0]
 [   0   13    0    0]]


In [40]:
from sklearn.metrics import classification_report

print(
    classification_report(
        y_val_encoded,
        val_predictions_no_weights,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

                     precision    recall  f1-score   support

      GENUINE_EVENT       0.29      0.44      0.35         9
             NORMAL       0.96      1.00      0.98      3019
       SENSOR_FAULT       1.00      0.04      0.07       107
TRANSMISSION_GLITCH       0.00      0.00      0.00        13

           accuracy                           0.96      3148
          macro avg       0.56      0.37      0.35      3148
       weighted avg       0.96      0.96      0.94      3148



- actual training class counts + current weights ek table mein

In [41]:
class_distribution = pd.DataFrame({
    "class_id": classes,
    "class_name": [
        label_encoder.inverse_transform([c])[0]
        for c in classes
    ],
    "training_count": [
        np.sum(y_train_encoded == c)
        for c in classes
    ],
    "current_weight": [
        class_weights[c]
        for c in classes
    ]
})

class_distribution

,class_id,class_name,training_count,current_weight
0,0,GENUINE_EVENT,90,40.802778
1,1,NORMAL,13999,0.262322
2,2,SENSOR_FAULT,536,6.851213
3,3,TRANSMISSION_GLITCH,64,57.378906


In [42]:
# Soften the extreme class weights

soft_class_weights = {
    class_label: np.sqrt(weight)
    for class_label, weight in class_weights.items()
}

print("Softened class weights:")

for class_label, weight in soft_class_weights.items():
    class_name = label_encoder.inverse_transform([class_label])[0]

    print(
        f"{class_label} -> {class_name}: {weight:.4f}"
    )

Softened class weights:
0 -> GENUINE_EVENT: 6.3877
1 -> NORMAL: 0.5122
2 -> SENSOR_FAULT: 2.6175
3 -> TRANSMISSION_GLITCH: 7.5749


- third baseline model

In [43]:
# Build the third baseline model with softened class weights

model_soft_weights = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(10,)),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.20),
    tf.keras.layers.Dense(4, activation="softmax")
])

model_soft_weights.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

print("Third baseline model created.")

Third baseline model created.


- Train third model

In [44]:
# Train the third baseline model with softened class weights

history_soft_weights = model_soft_weights.fit(
    X_train_scaled,
    y_train_encoded,
    validation_data=(
        X_val_scaled,
        y_val_encoded
    ),
    epochs=30,
    batch_size=32,
    class_weight=soft_class_weights,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9290 - loss: 0.5452 - val_accuracy: 0.9597 - val_loss: 0.4075
Epoch 2/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9512 - loss: 0.4553 - val_accuracy: 0.9482 - val_loss: 0.3982
Epoch 3/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9511 - loss: 0.4370 - val_accuracy: 0.9473 - val_loss: 0.3732
Epoch 4/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9528 - loss: 0.4191 - val_accuracy: 0.9463 - val_loss: 0.3931
Epoch 5/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9520 - loss: 0.4193 - val_accuracy: 0.9396 - val_loss: 0.4051
Epoch 6/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9528 - loss: 0.4084 - val_accuracy: 0.9438 - val_loss: 0.3769
Epoch 7/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9532 - loss: 0.4014 - val_accuracy: 0.9492 - val_loss: 0.3679
Epoch 8/30
460/460 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9530 - loss: 0.3960 - val_accuracy: 0.

- soft-weight model validation prediction

In [45]:
# Predict classes for the validation set
val_probabilities_soft = model_soft_weights.predict(
    X_val_scaled,
    verbose=0
)

val_predictions_soft = np.argmax(
    val_probabilities_soft,
    axis=1
)

print("Validation prediction distribution:")
print(
    pd.Series(
        val_predictions_soft
    ).value_counts().sort_index()
)

print("\nActual validation distribution:")
print(
    pd.Series(
        y_val_encoded
    ).value_counts().sort_index()
)

print("\nConfusion matrix:")
print(
    confusion_matrix(
        y_val_encoded,
        val_predictions_soft
    )
)

Validation prediction distribution:
0      37
1    3064
2      47
Name: count, dtype: int64

Actual validation distribution:
0       9
1    3019
2     107
3      13
Name: count, dtype: int64

Confusion matrix:
[[   7    1    1    0]
 [  28 2956   35    0]
 [   1   96   10    0]
 [   1   11    1    0]]


- Phir classification report

In [46]:
print(
    classification_report(
        y_val_encoded,
        val_predictions_soft,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

                     precision    recall  f1-score   support

      GENUINE_EVENT       0.19      0.78      0.30         9
             NORMAL       0.96      0.98      0.97      3019
       SENSOR_FAULT       0.21      0.09      0.13       107
TRANSMISSION_GLITCH       0.00      0.00      0.00        13

           accuracy                           0.94      3148
          macro avg       0.34      0.46      0.35      3148
       weighted avg       0.93      0.94      0.94      3148



#### small comparison table (M1, M2, M3)

In [47]:
from sklearn.metrics import precision_score, recall_score, f1_score

# Collect validation predictions from all three baseline models

# Model 1: Extreme class weights
val_probabilities_weighted = model.predict(
    X_val_scaled,
    verbose=0
)

val_predictions_weighted = np.argmax(
    val_probabilities_weighted,
    axis=1
)

# Model 2: No class weights
val_probabilities_no_weights = model_no_weights.predict(
    X_val_scaled,
    verbose=0
)

val_predictions_no_weights = np.argmax(
    val_probabilities_no_weights,
    axis=1
)

# Model 3: Softened class weights
val_probabilities_soft = model_soft_weights.predict(
    X_val_scaled,
    verbose=0
)

val_predictions_soft = np.argmax(
    val_probabilities_soft,
    axis=1
)


# Create a comparison table
results = []

models = {
    "Extreme Weights": val_predictions_weighted,
    "No Weights": val_predictions_no_weights,
    "Soft Weights": val_predictions_soft,
}

for model_name, predictions in models.items():

    results.append({
        "Model": model_name,
        "Accuracy": (
            predictions == y_val_encoded
        ).mean(),
        "Macro Precision": precision_score(
            y_val_encoded,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro Recall": recall_score(
            y_val_encoded,
            predictions,
            average="macro",
            zero_division=0
        ),
        "Macro F1": f1_score(
            y_val_encoded,
            predictions,
            average="macro",
            zero_division=0
        )
    })

comparison_df = pd.DataFrame(results)

comparison_df.round(4)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1
0,Extreme Weights,0.4352,0.2636,0.4200,0.1923
1,No Weights,0.9590,0.5619,0.3698,0.3498
2,Soft Weights,0.9444,0.3417,0.4626,0.3515


- Macro recall and Macro F1

- Now moving to LSTM encoder hoping that would help us in this Time- series sequential data.